In [ ]:
# use if drawing of snippets stored in excel and adding more

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load the Excel file
df = pd.read_excel("../data/senarios.xlsx")

In [ ]:

def create_entry_form(df):
    # Dropdown for previous entries
    previous_entry_dropdown = widgets.Dropdown(
        options=["New Entry"] + list(df["scenario_id"]),
        description="Load Entry:"
    )

    # Form fields
    scenario_id = widgets.Text(description="Scenario ID")
    case_id = widgets.Text(description="Case ID")
    snippet_id = widgets.Text(description="Snippet ID")
    scenario_type = widgets.Text(description="Scenario Type")
    protected_attr = widgets.Text(description="Protected Attr")
    protected_attr_group = widgets.Text(description="Attr Group")
    protected_value = widgets.Text(description="Protected Value (comma-separated)")
    text_snippet = widgets.Textarea(description="Text Snippet")
    change_summary = widgets.Textarea(description="Change Summary (comma-separated)")

    submit_button = widgets.Button(description="Submit", button_style='success')
    output = widgets.Output()

    # Autofill when selecting previous entry
    def load_entry(change):
        if previous_entry_dropdown.value != "New Entry":
            entry = df[df["scenario_id"] == previous_entry_dropdown.value].iloc[0]
            scenario_id.value = entry["scenario_id"]
            case_id.value = entry["case_id"]
            snippet_id.value = entry["snippet_id"]
            scenario_type.value = entry["scenario_type"]
            protected_attr.value = entry["protected_attr"]
            protected_attr_group.value = entry["protected_attr_group"]
            protected_value.value = ", ".join(entry["protected_value"])
            text_snippet.value = entry["text_snippet"]
            change_summary.value = ", ".join(entry["change_summary"])
        else:
            # Clear fields for new entry
            scenario_id.value = ""
            case_id.value = ""
            snippet_id.value = ""
            scenario_type.value = ""
            protected_attr.value = ""
            protected_attr_group.value = ""
            protected_value.value = ""
            text_snippet.value = ""
            change_summary.value = ""

    previous_entry_dropdown.observe(load_entry, names='value')

    # Submit logic
    
    def on_submit(b):
        new_entry = {
            "scenario_id": scenario_id.value,
            "case_id": case_id.value,
            "snippet_id": snippet_id.value,
            "scenario_type": scenario_type.value,
            "protected_attr": protected_attr.value,
            "protected_attr_group": protected_attr_group.value,
            "protected_value": [v.strip() for v in protected_value.value.split(",")],
            "text_snippet": text_snippet.value,
            "change_summary": [c.strip() for c in change_summary.value.split(",")]
        }

        df.loc[len(df)] = new_entry

        with output:
            output.clear_output()
            print("✅ Entry added")
            display(df)


    submit_button.on_click(on_submit)

    form = widgets.VBox([
        previous_entry_dropdown,
        scenario_id, case_id, snippet_id, scenario_type,
        protected_attr, protected_attr_group, protected_value,
        text_snippet, change_summary,
        submit_button, output
    ])

    display(form)

    return df


In [ ]:
# Call the function
df = create_entry_form(df)

In [ ]:
df.head(10)

In [ ]:
output_file = "../data/senarios.xlsx"

df.to_excel(
    output_file,
    index=False,
    engine="openpyxl"
)